# Deliberately overfit ACS 2018 target
Run manually from the project root with the project .venv kernel. No experiments have been executed. This is a separate research condition; the original experiments_2018.ipynb remains unchanged.

Fixed settings: 20,000 training records, hidden layers (512, 256, 128), zero dropout, zero weight decay, 100 epochs, no early stopping, and no best-checkpoint restoration. The returned model is the final epoch. These settings encourage overfitting but do not guarantee it or any particular membership-attack accuracy.

Using baseline seed 42, preserve the original 10,000 validation, 10,000 test, 10,000 MIA non-members, and 10,000 MIA members. All designated MIA members remain in the reduced training set; another 10,000 training records are sampled. The other 145,665 baseline training records are unused. A fresh preprocessor fits on these 20,000 training records only. If you changed the baseline split settings, update SPLIT_CONFIG here to match.

Validation is monitored, not used for checkpoint selection in this condition. Keep test and MIA results out of configuration selection. This condition changes training size, capacity, regularization, and duration together; it cannot attribute any observed effect to one change alone. Later ablations can isolate those factors. No saving or attack training is performed by these cells.

In [21]:
import importlib
import pandas as pd
import src.data
import src.training
importlib.reload(src.data)
importlib.reload(src.training)
from src.data import load_acs_income, split_acs_data, subset_target_training, preprocess_partitions
from src.training import train_target_model, evaluate_model

OVERFIT_SEED = 42
OVERFIT_TRAIN_SIZE = 150_000
SPLIT_CONFIG = dict(validation_size=10_000, test_size=10_000,
                   mia_nonmember_size=10_000, mia_member_size=10_000, seed=OVERFIT_SEED)
OVERFIT_CONFIG = dict(
    hidden_dims=(512, 256, 128), dropout=0.0, epochs=100,
    batch_size=256, learning_rate=1e-3, weight_decay=0.0,
    seed=OVERFIT_SEED, device='cpu', early_stopping=False,
    restore_best=False, monitor_train=True,
)

In [22]:
overfit_source = load_acs_income('data/processed/acs_income_ca_2018.csv')
baseline_partitions_for_ids = split_acs_data(overfit_source, **SPLIT_CONFIG)
overfit_partitions = subset_target_training(
    baseline_partitions_for_ids, train_size=OVERFIT_TRAIN_SIZE, seed=OVERFIT_SEED)
print('Unused baseline training records:',
      len(baseline_partitions_for_ids['train']) - len(overfit_partitions['train']))
pd.DataFrame({name: {'records': len(frame), 'positive_rate': frame['income_gt_50k'].mean()}
              for name, frame in overfit_partitions.items()}).T

Unused baseline training records: 15665


,records,positive_rate
validation,10000.0,0.410600
test,10000.0,0.410600
mia_nonmembers,10000.0,0.410600
train,150000.0,0.410567
mia_members,10000.0,0.410600


In [23]:
# Fit ONLY on the reduced training set. Do not reuse the baseline encoder.
overfit_preprocessor, overfit_arrays = preprocess_partitions(overfit_partitions)
{name: values['X'].shape for name, values in overfit_arrays.items()}

{'validation': (10000, 815),
 'test': (10000, 815),
 'mia_nonmembers': (10000, 815),
 'train': (150000, 815),
 'mia_members': (10000, 815)}

In [24]:
# This cell trains when YOU run it. A large CPU model may take time.
overfit_model, overfit_history, overfit_metadata = train_target_model(
    overfit_arrays['train']['X'], overfit_arrays['train']['y'],
    overfit_arrays['validation']['X'], overfit_arrays['validation']['y'],
    **OVERFIT_CONFIG)
print('Returned epoch:', overfit_metadata['returned_epoch'])
pd.DataFrame(overfit_history)

Epoch 001: train loss=0.3998, validation loss=0.3783, validation accuracy=0.8321
Epoch 002: train loss=0.3743, validation loss=0.3763, validation accuracy=0.8321
Epoch 003: train loss=0.3619, validation loss=0.3786, validation accuracy=0.8297
Epoch 004: train loss=0.3454, validation loss=0.3884, validation accuracy=0.8277
Epoch 005: train loss=0.3235, validation loss=0.4004, validation accuracy=0.8243
Epoch 006: train loss=0.2978, validation loss=0.4364, validation accuracy=0.8234
Epoch 007: train loss=0.2703, validation loss=0.4693, validation accuracy=0.8176
Epoch 008: train loss=0.2441, validation loss=0.5555, validation accuracy=0.8105
Epoch 009: train loss=0.2212, validation loss=0.6603, validation accuracy=0.8107
Epoch 010: train loss=0.2013, validation loss=0.6800, validation accuracy=0.8031
Epoch 011: train loss=0.1836, validation loss=0.7865, validation accuracy=0.8036
Epoch 012: train loss=0.1692, validation loss=0.8969, validation accuracy=0.8002
Epoch 013: train loss=0.1556

,epoch,train_loss,validation_accuracy,validation_roc_auc,validation_log_loss,train_eval_accuracy,train_eval_roc_auc,train_eval_log_loss
0,1,0.399777,0.8321,0.908157,0.378270,0.831667,0.911236,0.371572
1,2,0.374264,0.8321,0.908649,0.376318,0.839613,0.918652,0.357803
2,3,0.361924,0.8297,0.907316,0.378575,0.848700,0.925847,0.341631
3,4,0.345438,0.8277,0.906243,0.388371,0.860007,0.936408,0.317132
4,5,0.323547,0.8243,0.899850,0.400419,0.873133,0.947969,0.293944
...,...,...,...,...,...,...,...,...
95,96,0.028257,0.7864,0.854881,1.888207,0.989653,0.999664,0.023318
96,97,0.027682,0.7882,0.853513,1.883533,0.989193,0.999593,0.025590
97,98,0.029767,0.7879,0.854481,1.901641,0.988760,0.999601,0.025966
98,99,0.028266,0.7911,0.854817,1.866413,0.988293,0.999510,0.028428


## Inspect overfitting
Compare end-of-epoch training accuracy/loss against validation. train_loss is averaged across updates during the epoch; train_eval_log_loss evaluates the final weights for that epoch, so it is the directly comparable loss. A growing gap supports overfitting; a high training accuracy alone does not.

In [25]:
overfit_learning_curve = pd.DataFrame(overfit_history)
overfit_learning_curve['accuracy_gap_pp'] = 100 * (
    overfit_learning_curve['train_eval_accuracy'] - overfit_learning_curve['validation_accuracy'])
overfit_learning_curve['validation_minus_train_loss'] = (
    overfit_learning_curve['validation_log_loss'] - overfit_learning_curve['train_eval_log_loss'])
overfit_learning_curve[['epoch', 'train_eval_accuracy', 'validation_accuracy',
    'accuracy_gap_pp', 'train_eval_log_loss', 'validation_log_loss',
    'validation_minus_train_loss']].iloc[::10].copy()

,epoch,train_eval_accuracy,validation_accuracy,accuracy_gap_pp,train_eval_log_loss,validation_log_loss,validation_minus_train_loss
0,1,0.831667,0.8321,-0.043333,0.371572,0.378270,0.006698
10,11,0.931713,0.8036,12.811333,0.159082,0.786466,0.627384
20,21,0.967353,0.7920,17.535333,0.077191,1.311392,1.234201
30,31,0.975793,0.7886,18.719333,0.056767,1.507580,1.450813
40,41,0.981187,0.7891,19.208667,0.045041,1.661348,1.616307
50,51,0.984653,0.7927,19.195333,0.035799,1.759042,1.723243
60,61,0.985627,0.7885,19.712667,0.033795,1.787340,1.753544
70,71,0.987113,0.7862,20.091333,0.029774,1.846422,1.816648
80,81,0.987720,0.7882,19.952000,0.028305,1.866463,1.838158
90,91,0.989000,0.7902,19.880000,0.024868,1.877895,1.853027


## Final income-prediction evaluation
Evaluate once after completing the fixed training budget. Do not choose an epoch or configuration using this test result. MIA evaluation is a separate next step, using overfit_model and overfit_arrays (never the old baseline arrays).

In [26]:
overfit_metrics = {name: evaluate_model(overfit_model, values['X'], values['y'])
    for name, values in overfit_arrays.items() if name in ['train', 'validation', 'test']}
display(pd.DataFrame(overfit_metrics).T)
print('Train-validation accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['validation']['accuracy']))
print('Train-test accuracy gap (percentage points):',
      100 * (overfit_metrics['train']['accuracy'] - overfit_metrics['test']['accuracy']))

,accuracy,roc_auc,log_loss
validation,0.788100,0.856384,1.941775
test,0.795600,0.861833,1.877053
train,0.990347,0.999717,0.021427


Train-validation accuracy gap (percentage points): 20.224666666666668
Train-test accuracy gap (percentage points): 19.474666666666675


## Membership attack on the overfit target
Run after overfit target training. Use five diverse Adult shadows (XGBoost, random forest, MLP, decision tree, logistic regression), then train a fixed RANDOM FOREST membership classifier on class-scaled logits, blind walk, entropy, and cross-entropy. The target is unchanged. See data/paper_attack_features.md for formulas, settings, and reproduction limits.

If original shadow_results is present, reuse it; otherwise running the shadow cell trains all five. Re-extract features from fitted shadows: the old seven-column attack_data cannot supply blind-walk measurements. Sample 2,000 members and 2,000 non-members per shadow; evaluate all 20,000 target MIA records. No target or 2017 records enter shadow/attack training. Income labels are assumed known. Record-ID groups keep repeated Adult records together; thresholds use Adult validation only.

Target queries use overfit_model, overfit_preprocessor, and native overfit_partitions (10,000 members and 10,000 non-members). Blind walk uses valid per-field tabular values and a fixed budget, an adaptation because the paper leaves noise details unspecified. Defaults are 20 steps and two walks, with maximum combined query budget 1,640,000 record predictions. Censored walks are reported. RF uses 300 trees, depth 12, minimum leaf size 20, square-root feature subsampling, and seed 42; no RF search is performed. No target results select thresholds.

In [ ]:
import src.shadow_models
import src.membership_attack
import src.paper_attack_features
importlib.reload(src.shadow_models)
importlib.reload(src.membership_attack)
importlib.reload(src.paper_attack_features)
from src.shadow_models import load_adult_data, train_shadow_models, summarize_shadow_models
from src.membership_attack import train_membership_classifier, evaluate_attack
from src.paper_attack_features import collect_shadow_paper_features, collect_target_paper_features
WALK_CONFIG = dict(max_steps=20, repeats=2, categorical_change_probability=0.05,
                   batch_size=512, seed=42)
SHADOW_RECORDS_PER_CLASS = 2000

In [ ]:
# Reuse the original diverse shadow pool if available in this kernel.
# Otherwise running this cell trains five diverse shadows, not matched MLPs.
if 'shadow_results' in globals():
    expected_algorithms = {'xgboost', 'random_forest', 'mlp', 'decision_tree', 'logistic_regression'}
    if set(shadow_results) != expected_algorithms:
        raise ValueError('shadow_results must contain the original five diverse algorithms.')
    overfit_shadow_results = shadow_results
    print('Reusing original diverse shadow_results.')
else:
    overfit_adult_auxiliary = load_adult_data()
    overfit_shadow_results = train_shadow_models(overfit_adult_auxiliary, seed=42)
summarize_shadow_models(overfit_shadow_results)

In [ ]:
# Query the diverse shadows using the restored walk budget.
overfit_adult_for_queries = load_adult_data()
overfit_shadow_attack_data, shadow_walk_diagnostics = collect_shadow_paper_features(
    overfit_shadow_results, overfit_adult_for_queries,
    per_class=SHADOW_RECORDS_PER_CLASS, config=WALK_CONFIG, seed=42)
display(pd.DataFrame(shadow_walk_diagnostics).T)
overfit_attack_result = train_membership_classifier(
    overfit_shadow_attack_data, seed=42,
    algorithm='random_forest', feature_set='paper')
# Adult attack diagnostics, not target membership performance.
pd.DataFrame(overfit_attack_result['metrics']).T

In [ ]:
# Queries the target using the SAME feature-extraction protocol.
overfit_target_attack_data, target_walk_diagnostics = collect_target_paper_features(
    overfit_model, overfit_preprocessor, overfit_partitions, config=WALK_CONFIG, seed=42)
display(pd.Series(target_walk_diagnostics, name='target query diagnostics'))
overfit_target_mia_metrics, overfit_membership_scores = evaluate_attack(
    overfit_attack_result['classifier'], overfit_target_attack_data,
    threshold=overfit_attack_result['accuracy_threshold'],
    low_fpr_threshold=overfit_attack_result['low_fpr_threshold'])
pd.DataFrame({
    'Adult attack test': overfit_attack_result['metrics']['test'],
    'Overfit ACS 2018 target MIA': overfit_target_mia_metrics,
}).T